# Startup Simulator - RL Training (Colab)

Train the AI model for the Startup Simulator board game using SB3 MaskablePPO.

**Steps:**
1. Clone the repo and install dependencies
2. Train with frozen-pool self-play
3. Evaluate against baselines
4. Export ONNX model for the web app
5. Download the trained model

## 1. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/ue-too/startup-tabletop.git
%cd startup-tabletop

# Install dependencies
!pip install -e ".[dev]" sb3-contrib gymnasium onnx onnxruntime -q

In [ ]:
# Verify installation
import torch
torch.distributions.Distribution.set_default_validate_args(False)

from sb3_contrib import MaskablePPO
from startup_simulator.engine import GameEngine
from training.selfplay_env import SelfPlayEnv

# Quick smoke test
engine = GameEngine(num_players=2, seed=42)
print(f'Engine works: {len(engine.get_legal_actions())} legal actions')

env = SelfPlayEnv(num_players=2, seed=42)
obs, info = env.reset()
print(f'Env works: obs shape={obs["observation"].shape}')
print('All good!')

## 2. Train with Frozen-Pool Self-Play

Adjust `TIMESTEPS` based on how long you want to train:
- **1M steps** (~5 min): decent model, ~90% vs random
- **2M steps** (~10 min): strong model, ~95%+ vs random
- **5M steps** (~25 min): very strong
- **10M steps** (~50 min): maximum within Colab session

In [ ]:
import os, time
import torch
torch.distributions.Distribution.set_default_validate_args(False)

from sb3_contrib import MaskablePPO
from training.frozen_pool_env import FrozenPoolEnv
from training.callbacks import SelfPlayCallback

# === CONFIGURATION ===
TIMESTEPS = 2_000_000      # Total training steps
POOL_INTERVAL = 200_000    # Save to pool every N steps
LEARNING_RATE = 3e-4
# =====================

pool_dir = 'checkpoints/pool'
save_dir = 'checkpoints/model'
os.makedirs(pool_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

# Seed the pool with a random model
if not any(f.endswith('.zip') for f in os.listdir(pool_dir)):
    env_tmp = FrozenPoolEnv(seed=0, pool_dir=pool_dir)
    init_model = MaskablePPO('MultiInputPolicy', env_tmp, verbose=0)
    init_model.save(os.path.join(pool_dir, 'pool_000_random.zip'))
    del init_model, env_tmp
    print('Pool seeded with random model')

# Create environment and model
env = FrozenPoolEnv(seed=42, reward_mode='shaped', pool_dir=pool_dir)
model = MaskablePPO(
    'MultiInputPolicy', env,
    n_steps=512, batch_size=128, n_epochs=4,
    learning_rate=LEARNING_RATE, gamma=0.99,
    ent_coef=0.01, max_grad_norm=0.5, verbose=0,
)

callback = SelfPlayCallback(
    save_dir=save_dir, save_freq=POOL_INTERVAL,
    eval_freq=POOL_INTERVAL, eval_games=30, verbose=1,
)

print(f'Training {TIMESTEPS:,} steps with frozen-pool self-play...')
start = time.time()
trained = 0
pool_counter = 1

while trained < TIMESTEPS:
    chunk = min(POOL_INTERVAL, TIMESTEPS - trained)
    model.learn(total_timesteps=chunk, callback=callback, reset_num_timesteps=False)
    trained += chunk
    
    # Save current model to opponent pool
    pool_path = os.path.join(pool_dir, f'pool_{pool_counter:03d}_{trained//1000}k.zip')
    model.save(pool_path)
    pool_counter += 1
    
    elapsed = time.time() - start
    pool_size = len([f for f in os.listdir(pool_dir) if f.endswith('.zip')])
    print(f'  [{trained//1000}k/{TIMESTEPS//1000}k] Pool: {pool_size} models, {elapsed:.0f}s')

model.save(os.path.join(save_dir, 'model_final'))
elapsed = time.time() - start
print(f'\nDone in {elapsed:.0f}s ({TIMESTEPS/elapsed:.0f} steps/sec)')

## 3. Evaluate

In [ ]:
from training.evaluate import evaluate_agents, make_heuristic_agent, make_random_agent, make_sb3_agent

trained_agent = make_sb3_agent('checkpoints/model/model_final')
random_ag = make_random_agent(42)
heuristic = make_heuristic_agent(42)

GAMES = 200

def win_rate(a, b, games=GAMES):
    r1 = evaluate_agents([a, b], num_games=games)
    r2 = evaluate_agents([b, a], num_games=games)
    wr = (r1['wins'][0] + r2['wins'][1]) / (games * 2)
    avg = (r1['avg_scores'][0] + r2['avg_scores'][1]) / 2
    return wr, avg

print(f'=== Evaluation ({GAMES*2} games each) ===')
wr, avg = win_rate(trained_agent, random_ag)
print(f'Trained vs Random:    {wr:.0%}  (avg score: {avg:.1f})')
wr, avg = win_rate(trained_agent, heuristic)
print(f'Trained vs Heuristic: {wr:.0%}  (avg score: {avg:.1f})')
wr, _ = win_rate(heuristic, random_ag)
print(f'Heuristic vs Random:  {wr:.0%}  (baseline)')

## 4. Export to ONNX (for web app)

In [ ]:
import torch
import numpy as np
from sb3_contrib import MaskablePPO

model = MaskablePPO.load('checkpoints/model/model_final')
policy = model.policy

OBS_SIZE, MAX_ACTIONS = 2800, 512

class PolicyWrapper(torch.nn.Module):
    def __init__(self, p):
        super().__init__()
        self.fe = p.features_extractor
        self.mlp = p.mlp_extractor
        self.act = p.action_net
    def forward(self, obs, mask):
        f = self.fe({'observation': obs, 'action_mask': mask})
        pi, _ = self.mlp(f)
        logits = self.act(pi)
        return logits + (mask - 1.0) * 1e8

wrapper = PolicyWrapper(policy)
wrapper.eval()
dummy_obs = torch.zeros(1, OBS_SIZE)
dummy_mask = torch.ones(1, MAX_ACTIONS)

output_path = 'checkpoints/ai_model.onnx'
torch.onnx.export(
    wrapper, (dummy_obs, dummy_mask), output_path,
    input_names=['observation', 'action_mask'],
    output_names=['logits'],
    dynamic_axes={'observation': {0: 'b'}, 'action_mask': {0: 'b'}, 'logits': {0: 'b'}},
    opset_version=17,
)

# Verify
import onnxruntime as ort
session = ort.InferenceSession(output_path)
ort_out = session.run(None, {'observation': dummy_obs.numpy(), 'action_mask': dummy_mask.numpy()})
with torch.no_grad():
    pt_out = wrapper(dummy_obs, dummy_mask)
diff = np.abs(pt_out.numpy() - ort_out[0]).max()
size_mb = os.path.getsize(output_path) / (1024*1024)
print(f'ONNX exported: {size_mb:.1f} MB, max diff: {diff:.6f}')

## 5. Download

In [ ]:
from google.colab import files

# Download the ONNX model (for the web app)
files.download('checkpoints/ai_model.onnx')

# Optionally download the SB3 checkpoint (for continued training)
# files.download('checkpoints/model/model_final.zip')

## Optional: Continue Training from Checkpoint

If you want to continue training in a new session, upload the model and pool:

In [ ]:
# # Upload a previous checkpoint to continue training
# from google.colab import files
# uploaded = files.upload()  # Upload model_final.zip
# 
# import shutil
# shutil.move('model_final.zip', 'checkpoints/model/model_final.zip')
# 
# # Load and continue
# env = FrozenPoolEnv(seed=42, reward_mode='shaped', pool_dir='checkpoints/pool')
# model = MaskablePPO.load('checkpoints/model/model_final', env=env)
# model.learn(total_timesteps=1_000_000)  # Train more